# cTWAS via the three-step pecotmr API

## Description

Drives a multi-LD-block cTWAS analysis through `pecotmr`'s split pipeline:

- `[ctwas_1]` &mdash; `assembleCtwasInputs`: consume the block-grid manifest + the flat weight set (`--twas-weights`), placing each gene into its home LD block from its `region` provenance, and produce the assembled ctwas inputs (`z_snp`, `weights`, `region_info`, `snp_map`, `LD_map`, LD/snpInfo loader closures).
- `[ctwas_2]` &mdash; `estCtwasParam`: estimate the joint group prior + prior variance (`ctwas::est_param`). `--fallback-to-prefit` recovers from accurate-EM NaN divergence by re-running the prefit step only (mirrors the legacy ctwas_2 workaround).
- `[ctwas_3]` &mdash; `screenCtwasRegions` + `finemapCtwasRegions` &rarr; `asCtwasResult`: screen regions, fine-map causal genes, and structure the result as a `CtwasResult` (one row per (gwasStudy, study, context, method)); optional `--param-override` substitutes hand-tuned priors before screening.

The three steps chain via intermediate RDS files (`{name}.ctwas_inputs.rds`, `{name}.ctwas_est.rds`, `{name}.ctwas_finemap.rds`) so individual stages can be re-run independently.

## Inputs

- `--manifest` &mdash; block-grid manifest TSV with columns:
    - `region_id` (unique string)
    - `gwas_sumstats_rds` (path to per-block `GwasSumStats` RDS from `gwas_sumstats_construct.R`)
- `--twas-weights` &mdash; comma-separated FLAT per-gene weight RDS (`TwasWeights` or `QtlFineMappingResult`); placed into home LD blocks by pecotmr from each gene's `region`.
- `--fine-mapping-results` &mdash; optional comma-separated `FineMappingResult` RDS, used only as the CS/PIP rescue-filter source.
- Optional weight pre-filters (`--twas-weight-cutoff`, `--cs-min-cor`, `--min-pip-cutoff`, `--max-num-variants`).
- Optional `--twas-z` precomputed TWAS-Z `GRanges` RDS (output of `twas.ipynb`).
- Step 2 knobs: `--thin`, `--niter-prefit`, `--niter`, `--min-group-size`, `--min-p-single-effect`, `--fallback-to-prefit`.
- Step 3 knobs: `--L`, `--no-filter-L`, `--min-nonsnp-pip`, `--param-override`, `--keep-snps`.

## Output

- `{cwd}/{name}.ctwas_inputs.rds` (step 1)
- `{cwd}/{name}.ctwas_est.rds` (step 2)
- `{cwd}/{name}.ctwas_finemap.rds` (step 3; a `CtwasResult`)


## Example

```bash
sos run pipeline/ctwas.ipynb ctwas_3 \
    --cwd output --modular-script-dir /path/to/code/script \
    --name protocol_example_twas_chr22 \
    --manifest /tmp/gwas_smoke/ctwas_manifest_blessed.tsv \
    --fallback-to-prefit \
    --min-group-size 1
```


In [ ]:
[global]
parameter: cwd = path('output')
parameter: name = str
parameter: manifest = path
# --- step 1 (assemble) -----------------------------------------------
# flat weight source (comma-joined RDS): TwasWeights or QtlFineMappingResult;
# pecotmr places each gene into its home LD block from the weight's `region`
parameter: twas_weights = []
# optional flat FineMappingResult RDS used only as the CS/PIP filter source
parameter: fine_mapping_results = []
parameter: method = ''
parameter: twas_z = path('.')
parameter: twas_weight_cutoff = 0.0
parameter: cs_min_cor = 0.8
parameter: min_pip_cutoff = 0.0
parameter: max_num_variants = 'Inf'
# --- step 2 (est_param) ----------------------------------------------
parameter: thin = 0.1
parameter: niter_prefit = 3
parameter: niter = 30
parameter: group_prior_var_structure = 'shared_type'
parameter: min_group_size = 100
parameter: min_p_single_effect = 0.8
parameter: fallback_to_prefit = False
# --- step 3 (screen + finemap) ---------------------------------------
parameter: L = 5
# retain the SNP background as a dedicated study=context="SNP" CtwasResult row
parameter: keep_snps = False
parameter: no_filter_L = False
parameter: min_nonsnp_pip = 0.5
parameter: param_override = path('.')
# --- infrastructure --------------------------------------------------
parameter: modular_script_dir = path('code/script')
parameter: container = ''
parameter: job_size = 1
parameter: walltime = '1h'
parameter: mem = '16G'
parameter: numThreads = 1


In [ ]:
[ctwas_1 (assemble inputs)]
input: manifest
output: f"{cwd}/{name}.ctwas_inputs.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/ctwas_assemble.R \
        --manifest ${_input} \
        --twas-weights "${",".join([str(w) for w in twas_weights])}" \
        ${'--fine-mapping-results ' + ",".join([str(f) for f in fine_mapping_results]) if fine_mapping_results else ''} \
        ${'--method ' + method if method else ''} \
        ${'--twas-z ' + str(twas_z) if twas_z.is_file() else ''} \
        --twas-weight-cutoff ${twas_weight_cutoff} \
        --cs-min-cor ${cs_min_cor} \
        --min-pip-cutoff ${min_pip_cutoff} \
        --max-num-variants ${max_num_variants} \
        --output ${_output}


In [ ]:
[ctwas_2 (estimate priors)]
output: f"{cwd}/{name}.ctwas_est.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/ctwas_est.R \
        --inputs ${_input} \
        --thin ${thin} \
        --niter-prefit ${niter_prefit} \
        --niter ${niter} \
        --group-prior-var-structure ${group_prior_var_structure} \
        --min-group-size ${min_group_size} \
        --min-p-single-effect ${min_p_single_effect} \
        ${'--fallback-to-prefit' if fallback_to_prefit else ''} \
        --ncore ${numThreads} \
        --output ${_output}


In [ ]:
[ctwas_3 (screen + finemap)]
output: f"{cwd}/{name}.ctwas_finemap.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/ctwas_finemap.R \
        --est ${_input} \
        ${'--param-override ' + str(param_override) if param_override.is_file() else ''} \
        --L ${L} \
        ${'--no-filter-L' if no_filter_L else ''} \
        --min-nonsnp-pip ${min_nonsnp_pip} \
        ${'--keep-snps' if keep_snps else ''} \
        --ncore ${numThreads} \
        --output ${_output}
